# 📖 Lab 4: Efficiency — Deduplication + Crawler Traps

**Non-functional requirement:** *Crawl 10B pages efficiently in under 5 days.*

At 10B pages, wasting time on duplicates or infinite loops is catastrophic. Two key problems:
1. **URL dedup** — don't re-crawl URLs we've already seen
2. **Content dedup** — different URLs can serve identical content
3. **Crawler traps** — pages that generate infinite URLs

## Learning Objectives

- Implement URL-level deduplication
- Build content-level deduplication with hashing
- Compare DB index vs Bloom filter for dedup at scale
- Detect and prevent crawler traps with depth limiting

In [ ]:
import hashlib
import time
import os
from collections import deque
from urllib.parse import urlparse

print("✅ Ready.")

## 🔗 URL-Level Deduplication

Before adding a URL to the frontier, check if we've already crawled it. This is the first line of defense — prevents re-fetching the same URL.

In production, this check happens against the Metadata DB. Here we simulate with a set + normalize URLs (strip fragments, trailing slashes, lowercase domain).

In [ ]:
def normalize_url(url: str) -> str:
    """Normalize URL to catch obvious duplicates."""
    parsed = urlparse(url)
    # Lowercase scheme + domain, strip trailing slash, remove fragment
    normalized = f"{parsed.scheme.lower()}://{parsed.netloc.lower()}{parsed.path.rstrip('/')}"
    if parsed.query:
        normalized += f"?{parsed.query}"
    return normalized


class URLDeduplicator:
    """URL-level dedup using a set (in production: Metadata DB check)."""

    def __init__(self):
        self.seen: set[str] = set()
        self.stats = {"total": 0, "unique": 0, "duplicate": 0}

    def is_new(self, url: str) -> bool:
        self.stats["total"] += 1
        normalized = normalize_url(url)
        if normalized in self.seen:
            self.stats["duplicate"] += 1
            return False
        self.seen.add(normalized)
        self.stats["unique"] += 1
        return True


# Demo: URLs that look different but are the same page
dedup = URLDeduplicator()

test_urls = [
    "https://Example.COM/page",
    "https://example.com/page",        # same — different case
    "https://example.com/page/",       # same — trailing slash
    "https://example.com/page#section", # same — just a fragment
    "https://example.com/other",       # different
    "http://example.com/page",         # different — http vs https
    "https://example.com/page",        # duplicate
]

print("🔗 URL-Level Deduplication:\n")
for url in test_urls:
    normalized = normalize_url(url)
    is_new = dedup.is_new(url)
    icon = "✅ NEW" if is_new else "⏭️  SKIP"
    print(f"  {icon}  {url:<45} → {normalized}")

print(f"\n  📊 Total: {dedup.stats['total']} | Unique: {dedup.stats['unique']} | Duplicates: {dedup.stats['duplicate']}")

## 📝 Content-Level Deduplication

URL dedup isn't enough. `http://example.com` and `http://www.example.com` are different URLs but serve identical content. We hash the content and check if we've seen that hash before.

In [ ]:
class ContentDeduplicator:
    """
    Content-level dedup using SHA-256 hashes.
    In production: hash stored in Metadata DB with an index, or Redis Bloom filter.
    """

    def __init__(self):
        self.seen_hashes: set[str] = set()
        self.stats = {"total": 0, "unique": 0, "duplicate": 0}

    def compute_hash(self, content: str) -> str:
        return hashlib.sha256(content.encode()).hexdigest()

    def is_new_content(self, content: str) -> tuple[bool, str]:
        self.stats["total"] += 1
        content_hash = self.compute_hash(content)

        if content_hash in self.seen_hashes:
            self.stats["duplicate"] += 1
            return False, content_hash

        self.seen_hashes.add(content_hash)
        self.stats["unique"] += 1
        return True, content_hash


# Demo: different URLs, same content
content_dedup = ContentDeduplicator()

pages = [
    ("https://example.com",      "Welcome to Example Domain. This domain is for use in examples."),
    ("https://www.example.com",  "Welcome to Example Domain. This domain is for use in examples."),  # same content!
    ("https://other-site.com",   "Welcome to Example Domain. This domain is for use in examples."),  # also same!
    ("https://unique-site.com",  "This is a completely different page with unique content."),
    ("https://another.com",      "Welcome to Example Domain. This domain is for use in examples."),  # same again
]

print("📝 Content-Level Deduplication:\n")
for url, content in pages:
    is_new, content_hash = content_dedup.is_new_content(content)
    icon = "✅ NEW" if is_new else "⏭️  SKIP (duplicate content)"
    print(f"  {icon}")
    print(f"     URL:  {url}")
    print(f"     Hash: {content_hash[:16]}...")

print(f"\n  📊 Total: {content_dedup.stats['total']} | Unique content: {content_dedup.stats['unique']} | Duplicates: {content_dedup.stats['duplicate']}")
print(f"  💡 3 different URLs served identical content — skipped re-processing!")

## 🕳️ Crawler Traps: Infinite URL Generation

Some sites generate infinite unique URLs — calendar pages with endless date parameters, session IDs in URLs, etc. Without protection, our crawler would loop forever on a single domain.

**Fix:** Track **link depth** (hops from seed URL). Stop crawling at depth 15-20.

In [ ]:
MAX_DEPTH = 3       # low for demo (15-20 in production)
MAX_PAGES = 1000    # high enough that the depth limit trips first


def simulate_crawler_trap():
    """Simulates a site that generates infinite URLs (like a calendar trap)."""

    # Each page links to 3 "deeper" pages — fan-out of 3 per level.
    def get_links(url: str) -> list[str]:
        parsed = urlparse(url)
        path = parsed.path
        return [
            f"{parsed.scheme}://{parsed.netloc}{path}/page_a",
            f"{parsed.scheme}://{parsed.netloc}{path}/page_b",
            f"{parsed.scheme}://{parsed.netloc}{path}/page_c",
        ]

    frontier = deque()
    frontier.append(("https://trap-site.com", 0))  # (url, depth)
    visited = set()
    crawled = 0
    skipped_depth = 0

    print(f"🕳️  Crawler Trap Simulation (max depth = {MAX_DEPTH}):\n")

    while frontier and crawled < MAX_PAGES:
        url, depth = frontier.popleft()

        if url in visited:
            continue

        if depth > MAX_DEPTH:
            skipped_depth += 1
            if skipped_depth <= 3:
                print(f"  ⛔ Depth {depth}: SKIPPED {url}")
            elif skipped_depth == 4:
                print(f"  ⛔ ... (and more skipped at depth > {MAX_DEPTH})")
            continue

        visited.add(url)
        crawled += 1
        if crawled <= 10 or crawled % 20 == 0:
            print(f"  ✅ Depth {depth}: crawled {url}")

        for link in get_links(url):
            if link not in visited:
                frontier.append((link, depth + 1))

    # Without a depth limit, fan-out 3 at each level yields 3^depth URLs.
    # The trap is infinite — these numbers grow forever.
    would_grow_to = sum(3 ** d for d in range(MAX_DEPTH + 5))
    print(f"\n  📊 Crawled: {crawled} pages | Skipped (depth > {MAX_DEPTH}): {skipped_depth}")
    print(f"     Just 5 extra depth levels would add {would_grow_to:,}+ URLs — and it never stops.")
    print(f"     Depth limiting is what makes it safe to crawl unknown sites.")


simulate_crawler_trap()


## 🌸 Bloom Filter: Dedup at 10B-URL Scale

A Python `set` of 10B URLs would take **~1 TB of RAM** — we need a cheaper "have I seen this?" check.

A **Bloom filter** is a probabilistic set: it can say "definitely not seen" or "probably seen" using ~10 bits per item. For 10B URLs that's only ~12 GB — small enough to fit in a single Redis node.

- ✅ No false negatives — if it says "new", the URL really is new.
- ⚠️  Small false-positive rate — it may occasionally say "seen" when we haven't. In a crawler, the cost is just missing a page, which is acceptable at scale.

We'll hand-roll a tiny Bloom filter to show the trade-off. In production, use Redis' `BF.ADD` / `BF.EXISTS` (RedisBloom module).


In [ ]:
import math
from hashlib import blake2b


class BloomFilter:
    """Tiny educational Bloom filter. Not thread-safe."""

    def __init__(self, expected_items: int, false_positive_rate: float = 0.01):
        # Optimal bit-array size (m) and number of hash functions (k).
        m = -(expected_items * math.log(false_positive_rate)) / (math.log(2) ** 2)
        k = (m / expected_items) * math.log(2)
        self.m = int(math.ceil(m))
        self.k = max(1, int(round(k)))
        self.bits = bytearray((self.m + 7) // 8)

    def _hashes(self, item: str):
        # Derive k hash indices from one BLAKE2b digest (fast + good mixing).
        h = blake2b(item.encode(), digest_size=16).digest()
        h1 = int.from_bytes(h[:8], "big")
        h2 = int.from_bytes(h[8:], "big")
        for i in range(self.k):
            yield (h1 + i * h2) % self.m

    def add(self, item: str) -> None:
        for idx in self._hashes(item):
            self.bits[idx // 8] |= 1 << (idx % 8)

    def __contains__(self, item: str) -> bool:
        return all(
            self.bits[idx // 8] & (1 << (idx % 8))
            for idx in self._hashes(item)
        )


# Demo: insert 10k URLs, then check 10k unseen ones to measure false positives.
bf = BloomFilter(expected_items=10_000, false_positive_rate=0.01)

seen = {f"https://site.com/page/{i}" for i in range(10_000)}
for url in seen:
    bf.add(url)

# True positives: every inserted URL is reported as seen (guaranteed).
true_positive_hits = sum(1 for url in seen if url in bf)

# False positives: URLs we never inserted, but the filter thinks we did.
unseen = [f"https://other.com/page/{i}" for i in range(10_000)]
false_positive_hits = sum(1 for url in unseen if url in bf)

bits_per_item = bf.m / 10_000

print("🌸 Bloom filter results:\n")
print(f"  Size:            {bf.m:,} bits ({bf.m // 8:,} bytes) with {bf.k} hash functions")
print(f"  Bits per item:   {bits_per_item:.1f}  (a Python set would use ~1000+)")
print(f"  True positives:  {true_positive_hits}/10000   ← never missed a seen URL")
print(f"  False positives: {false_positive_hits}/10000 "
      f"(~{false_positive_hits/100:.1f}%, target was 1%)")
print("\n  💡 At 10B URLs, a set would need ~1 TB of RAM; this Bloom filter needs ~12 GB.")


## ✅ Summary

### Deduplication Strategy

```
Discovered URL → URL dedup (normalize + check DB) → already seen? SKIP
                                                   → new? → FETCH
Fetched content → Content dedup (SHA-256 hash) → same content? SKIP parsing
                                                → new? → PARSE + STORE
```

| Layer | What | Memory | False Positives |
|-------|------|--------|-----------------|
| **URL dedup** | Normalize URL + check Metadata DB | O(1) per lookup | None |
| **Content dedup (DB index)** ✅ | SHA-256 hash + indexed column | O(1) per lookup | None |
| **Content dedup (Bloom filter)** | Probabilistic set in Redis | O(1), very compact | Small % (configurable) |

### Crawler Trap Prevention

| Mechanism | How |
|-----------|-----|
| **Depth limiting** | Track hop count from seed URL. Stop at depth 15-20. |
| **URL pattern detection** | (Advanced) Detect repeating path patterns like `/a/b/a/b/a/b/...` |

### Scaling Math

```
10B pages × 2MB avg ÷ (3,750 pages/sec × 8 machines) ≈ 3.9 days ✅
```

Parser workers auto-scale based on processing queue depth (Lambda / ECS Fargate).